# 02 · Cleaning walkthrough (M2)

The raw defects listed in PRD 8.3, how each one was found and how it was resolved. The pipeline (`src/clean.py`) does this for real; this notebook shows the reasoning.

In [1]:
import sys, json, warnings
sys.path.insert(0, "../src"); warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from utils import CONFIG, path
raw = pd.read_csv(path("raw", CONFIG["raw_files"]["traffic"]))
print(f"{len(raw):,} raw rows")

178,468 raw rows


## 1. Impossible values have a backup in the same row

In [2]:
good = raw[raw.traffic_volume >= 0]
print("traffic_volume == vehicle_count on valid rows:", (good.traffic_volume == good.vehicle_count).mean())
bad = raw[(raw.traffic_volume < 0) | raw.traffic_volume.isna()]
print(f"{len(bad):,} negative or missing volumes; vehicle_count present on {bad.vehicle_count.notna().mean():.0%} of them")
bad[["traffic_volume", "vehicle_count"]].head()

traffic_volume == vehicle_count on valid rows: 1.0
4,628 negative or missing volumes; vehicle_count present on 100% of them


,traffic_volume,vehicle_count
6,NaN,260
96,NaN,1100
142,NaN,159
175,NaN,423
221,NaN,749


In [3]:
ok = raw[raw.avg_speed.between(1, 200)]
length = ok.travel_time * ok.avg_speed / 60
print("segment length per road (km): spread inside each road =", length.groupby(ok.road_id).std().max().round(4))
fast = raw[raw.avg_speed > 200]
print(f"{len(fast)} speeds above 200 km/h; their travel_time looks normal:", fast.travel_time.describe()[["mean", "50%"]].round(2).to_dict())

segment length per road (km): spread inside each road = 0.0022
237 speeds above 200 km/h; their travel_time looks normal: {'mean': 3.57, '50%': 3.07}


Travel time is length / speed with a fixed length per segment, so a broken speed reading can be recomputed exactly. Interpolating it would only be an estimate of a value we already have.

## 2. Missing windows

In [4]:
expected = raw.road_id.nunique() * 151 * 48
print(f"expected {expected:,} segment-windows, raw has {len(raw.drop_duplicates(['road_id','date','time'])):,} unique")
ts = pd.to_datetime(raw.date + " " + raw.time)
g = raw.assign(ts=ts).drop_duplicates(["road_id", "ts"]).sort_values(["road_id", "ts"])
step = g.groupby("road_id")["ts"].diff().dt.total_seconds().div(1800).dropna()
gaps = (step[step > 1] - 1).astype(int)
print("missing-run lengths (windows):", gaps.value_counts().sort_index().head(10).to_dict())

expected 181,200 segment-windows, raw has 176,701 unique


missing-run lengths (windows): {1: 4272, 2: 113}


A missing row breaks lag features silently: `shift(1)` would return the reading from an hour or more earlier. The pipeline rebuilds the full grid first, fills short gaps by interpolation and long gaps with the segment-and-window median, and flags every rebuilt row so it is never used as a training target.

## 3. The generated quality report

In [5]:
print(path("reports", "data_quality.md").read_text())

# FlowCast - Data Quality Report

Generated by the pipeline from `traffic_sensor_log.csv` (sha256 8f793f3643c8). Every number below is counted by the code, not typed in.

## Defects found and how each was resolved

| Defect (PRD 8.3) | Found | Resolution |
|---|---|---|
| Duplicate log rows | 1767 exact, 0 key-level | Dropped; for key duplicates the most complete row was kept |
| Negative or missing traffic_volume | 4587 | Recovered exactly from vehicle_count (equal on every valid row) |
| avg_speed above 200 km/h | 4579 | Recovered exactly as 60 x segment length / travel_time |
| Occupancy above 100% | 229 | Capped at 100 |
| Statistically extreme readings (robust z > 4.0 vs the same segment and time-of-day slot) | volume 26, speed 463, occupancy 225 | Flagged and kept: after the rule fixes these are mostly real events and incidents |
| Missing sensor windows (absent rows) | 4499 | Rebuilt on the full segment x 30-min grid, flagged is_imputed = 1 |
| Missing volume / speed / occupancy